# 🎙️ Garhwali TTS Training with Coqui VITS

Fine-tune a VITS model on Garhwali audio for Ghughuti AI voice synthesis.

**Prerequisites:**
- 50–100+ audio recordings from pahaditube.in/voice-recording (≥ 30 min total)
- Google Colab with **GPU runtime** (T4 free tier works)

**Pipeline:**
1. Install dependencies & verify GPU
2. Download / upload recordings
3. Audio quality check — resample, trim silence, validate
4. Build character set from Devanagari text
5. Create train / eval split & VITS config
6. Fine-tune VITS (from English VITS base or from scratch)
7. Test & listen
8. Export & deploy to HuggingFace

## 1️⃣ Setup & Install Dependencies

In [ ]:
# Check GPU availability
!nvidia-smi

# Install Coqui TTS (pinned for reproducibility)
!pip install -q TTS==0.22.0

# Audio processing
!pip install -q librosa soundfile pydub

# For uploading to HuggingFace later
!pip install -q huggingface_hub

print('✅ Installation complete!')

In [ ]:
# Verify GPU and imports
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected — go to Runtime → Change runtime type → GPU')

## 2️⃣ Download Recordings from PahadiTube

In [ ]:
import os, requests, json

# Create dataset directories
DATASET_DIR = '/content/garhwali_dataset'
WAV_DIR = os.path.join(DATASET_DIR, 'wavs')
os.makedirs(WAV_DIR, exist_ok=True)

# PahadiTube API endpoint
API_URL = 'https://pahaditube.in/api/tts'

# Fetch recordings metadata
print('Fetching recordings from PahadiTube...')
try:
    response = requests.get(f'{API_URL}/recordings', timeout=30)
    response.raise_for_status()
    data = response.json()
    recordings = data.get('recordings', [])
    print(f'Found {len(recordings)} recordings')
except Exception as e:
    print(f'API unavailable: {e}')
    print('\n→ Use the manual ZIP upload cell below instead.')
    recordings = []

In [ ]:
# Download & convert audio files to 22050 Hz mono WAV
from pydub import AudioSegment

metadata_lines = []
downloaded = 0

for rec in recordings:
    sid = rec.get('sentenceId', rec.get('id', ''))
    text = rec.get('text', '')
    url = rec.get('url', rec.get('audio_url', ''))
    if not sid or not text or not url:
        continue

    try:
        audio_response = requests.get(url, timeout=30)
        audio_response.raise_for_status()

        temp_path = f'/content/temp_{sid}.webm'
        with open(temp_path, 'wb') as f:
            f.write(audio_response.content)

        # Convert to WAV (22050 Hz, mono, 16-bit)
        audio = AudioSegment.from_file(temp_path)
        audio = audio.set_frame_rate(22050).set_channels(1).set_sample_width(2)

        wav_path = os.path.join(WAV_DIR, f'{sid}.wav')
        audio.export(wav_path, format='wav')

        metadata_lines.append(f'{sid}|{text}|{text}')
        downloaded += 1

        os.remove(temp_path)

        if downloaded % 10 == 0:
            print(f'  Downloaded {downloaded}/{len(recordings)}...')
    except Exception as e:
        print(f'  Skip {sid}: {e}')

# Save metadata (LJSpeech format: id|raw_text|normalized_text)
META_FILE = os.path.join(DATASET_DIR, 'metadata.csv')
with open(META_FILE, 'w', encoding='utf-8') as f:
    f.write('\n'.join(metadata_lines))

print(f'\n✅ Downloaded {downloaded} recordings')
print(f'📄 Metadata: {META_FILE} ({len(metadata_lines)} entries)')

### Alternative: Upload ZIP manually

If the API download doesn't work, upload a ZIP with this structure:
```
garhwali_dataset/
├── wavs/
│   ├── garhwali_001.wav   (22050 Hz, mono)
│   └── ...
└── metadata.csv           (format: id|text|text)
```
Each row in `metadata.csv` is `filename_without_ext|raw text|normalized text`.
For Garhwali both text columns can be the same Devanagari text.

In [ ]:
# Upload ZIP manually (run this cell → click the upload button)
from google.colab import files

print('Upload your garhwali_dataset.zip:')
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        !unzip -o "{filename}" -d /content/
        print(f'✅ Extracted {filename}')

# Re-point paths after upload
DATASET_DIR = '/content/garhwali_dataset'
WAV_DIR = os.path.join(DATASET_DIR, 'wavs')
META_FILE = os.path.join(DATASET_DIR, 'metadata.csv')

## 3️⃣ Validate & Clean Audio

In [ ]:
import os, librosa, soundfile as sf, numpy as np

wav_dir = WAV_DIR
metadata_file = META_FILE

wav_files = sorted([f for f in os.listdir(wav_dir) if f.endswith('.wav')])
print(f'WAV files found: {len(wav_files)}')

with open(metadata_file, 'r', encoding='utf-8') as f:
    meta_lines = [l.strip() for l in f if l.strip()]
print(f'Metadata entries: {len(meta_lines)}')

# Show sample
print('\nSample entries:')
for line in meta_lines[:5]:
    print(f'  {line}')

In [ ]:
# Audio quality report: duration, silence ratio, clipping
import librosa, numpy as np, os

durations = []
issues = []

for wav_file in wav_files:
    path = os.path.join(wav_dir, wav_file)
    try:
        y, sr = librosa.load(path, sr=22050)
        dur = len(y) / sr
        durations.append(dur)

        # Flag very short or very long clips
        if dur < 1.0:
            issues.append(f'  ⚠️  {wav_file}: too short ({dur:.1f}s)')
        elif dur > 15.0:
            issues.append(f'  ⚠️  {wav_file}: very long ({dur:.1f}s)')

        # Flag clipping (samples near ±1.0)
        clip_ratio = np.mean(np.abs(y) > 0.99)
        if clip_ratio > 0.01:
            issues.append(f'  ⚠️  {wav_file}: clipping detected ({clip_ratio*100:.1f}%)')
    except Exception as e:
        issues.append(f'  ❌ {wav_file}: failed to load — {e}')

total_dur = sum(durations)
print(f'Total audio duration : {total_dur/60:.1f} minutes')
print(f'Average per clip     : {np.mean(durations):.1f}s')
print(f'Shortest clip        : {min(durations):.1f}s')
print(f'Longest clip         : {max(durations):.1f}s')

if issues:
    print(f'\n⚠️  {len(issues)} issues found:')
    for iss in issues[:20]:
        print(iss)
else:
    print('\n✅ All audio files look clean!')

if total_dur < 20 * 60:
    print(f'\n⚠️  Only {total_dur/60:.0f} min of audio — aim for ≥ 30 min for decent quality.')
    print('   Record more at pahaditube.in/voice-recording')

In [ ]:
# Trim leading/trailing silence from every clip & peak-normalize
import librosa, soundfile as sf, numpy as np, os

trimmed_count = 0
for wav_file in wav_files:
    path = os.path.join(wav_dir, wav_file)
    y, sr = librosa.load(path, sr=22050)

    # Trim silence (threshold 20 dB below peak)
    yt, _ = librosa.effects.trim(y, top_db=20)

    # Peak-normalize to -1 dBFS
    peak = np.max(np.abs(yt))
    if peak > 0:
        yt = yt / peak * 0.9

    sf.write(path, yt, sr)
    trimmed_count += 1

print(f'✅ Trimmed & normalized {trimmed_count} files')

## 4️⃣ Build Character Set & Train/Eval Split

In [ ]:
# Build the Devanagari character set from the actual text data
import json, os, random

with open(metadata_file, 'r', encoding='utf-8') as f:
    meta_lines = [l.strip() for l in f if l.strip()]

# Extract unique characters from text column
all_text = ''.join(line.split('|')[1] for line in meta_lines)
chars = sorted(set(all_text))

# Keep printable chars (drop control chars)
chars = [c for c in chars if c.isprintable()]

print(f'Unique characters: {len(chars)}')
print(f'Character set: {repr(chr(32).join(chars))}')

# --- Train / Eval split (90/10) ---
random.seed(42)
random.shuffle(meta_lines)
split_idx = max(1, int(len(meta_lines) * 0.9))

train_lines = meta_lines[:split_idx]
eval_lines  = meta_lines[split_idx:]

train_file = os.path.join(DATASET_DIR, 'metadata_train.csv')
eval_file  = os.path.join(DATASET_DIR, 'metadata_eval.csv')

with open(train_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(train_lines))
with open(eval_file, 'w', encoding='utf-8') as f:
    f.write('\n'.join(eval_lines))

print(f'\nTrain set: {len(train_lines)} utterances → {train_file}')
print(f'Eval set : {len(eval_lines)} utterances → {eval_file}')

In [ ]:
# Create VITS training config with proper Devanagari character handling
from TTS.tts.configs.vits_config import VitsConfig
from TTS.config.shared_configs import CharactersConfig

OUTPUT_DIR = '/content/garhwali_output'

# Character config — uses the actual charset discovered above
character_config = CharactersConfig(
    characters_class='TTS.tts.utils.text.characters.Graphemes',
    pad='_',
    eos='~',
    bos='^',
    characters=''.join(chars),
    punctuations='।,.!?;:\'\"-–… ',
)

config = VitsConfig(
    run_name='garhwali-vits',
    run_description='Garhwali TTS — VITS fine-tune for Ghughuti AI',

    # Audio (must match your WAV files)
    audio={
        'sample_rate': 22050,
        'win_length': 1024,
        'hop_length': 256,
        'num_mels': 80,
        'mel_fmin': 0,
        'mel_fmax': None,
    },

    # Training
    batch_size=16,          # Lower to 8 if OOM on T4
    eval_batch_size=8,
    num_loader_workers=4,
    num_eval_loader_workers=2,
    epochs=1000,            # VITS needs many epochs for small datasets
    lr_gen=0.0002,
    lr_disc=0.0002,
    save_step=500,
    save_checkpoints=True,
    save_best_after=1000,
    print_step=50,
    plot_step=100,
    mixed_precision=True,   # Saves VRAM on T4

    # Text — grapheme-based (no phonemizer needed for Devanagari)
    use_phonemes=False,
    characters=character_config,

    # Datasets — LJSpeech pipe-delimited format
    datasets=[
        {
            'formatter': 'ljspeech',
            'meta_file_train': 'metadata_train.csv',
            'meta_file_val': 'metadata_eval.csv',
            'path': DATASET_DIR,
            'language': 'hi',
        }
    ],

    output_path=OUTPUT_DIR,
)

config.save_json('/content/garhwali_config.json')
print(f'✅ Config saved — {len(chars)} characters, {config.epochs} epochs')
print(f'   Batch size: {config.batch_size}  |  Mixed precision: {config.mixed_precision}')

## 5️⃣ Fine-tune VITS

**Two options:**
- **Option A** — Fine-tune from a pre-trained English VITS checkpoint (faster convergence, ~2-4 hrs on T4)
- **Option B** — Train from scratch (needs more data & epochs, ~6-12 hrs on T4)

Run **one** of the cells below.

In [ ]:
# ── Option A: Download pre-trained VITS for fine-tuning ──
# Downloads the English LJSpeech VITS checkpoint as starting weights.
# Mismatched layers (character embeddings) are automatically skipped.

from TTS.utils.manage import ModelManager
import shutil

manager = ModelManager()
model_path, config_path, _ = manager.download_model('tts_models/en/ljspeech/vits')
print(f'Base model: {model_path}')

BASE_CKPT = '/content/base_vits.pth'
shutil.copy(model_path, BASE_CKPT)
print(f'✅ Base checkpoint ready at {BASE_CKPT}')

In [ ]:
# ── Option A: Launch fine-tuning from pre-trained model ──
# --restore_path loads weights (mismatched layers are skipped automatically).

!python -m TTS.bin.train_tts \
    --config_path /content/garhwali_config.json \
    --restore_path /content/base_vits.pth

In [ ]:
# ── Option B: Train from scratch (no base model) ──
# Use this if you have > 2 hours of audio or want a pure Garhwali voice.

!python -m TTS.bin.train_tts \
    --config_path /content/garhwali_config.json

In [ ]:
# ── Resume training after Colab disconnects / interruption ──
# --continue_path points to the run folder (auto-finds latest checkpoint).

import glob
run_dirs = sorted(glob.glob('/content/garhwali_output/garhwali-vits-*'))
if run_dirs:
    print(f'Resuming from: {run_dirs[-1]}')
    !python -m TTS.bin.train_tts --continue_path {run_dirs[-1]}
else:
    print('No checkpoint found — run training first.')

## 6️⃣ Test the Fine-tuned Model

In [ ]:
# Find the best checkpoint
import os, glob

output_dir = '/content/garhwali_output/'

# Look inside the run subfolder
run_dirs = sorted(glob.glob(os.path.join(output_dir, 'garhwali-vits-*')))
run_dir = run_dirs[-1] if run_dirs else output_dir

best_candidates = glob.glob(os.path.join(run_dir, 'best_model*.pth'))
if best_candidates:
    best_model = best_candidates[0]
    print(f'✅ Best model: {best_model}')
else:
    all_ckpts = sorted(glob.glob(os.path.join(run_dir, 'checkpoint_*.pth')))
    best_model = all_ckpts[-1] if all_ckpts else None
    print(f'Using latest checkpoint: {best_model}')

config_file = os.path.join(run_dir, 'config.json')
print(f'Config: {config_file}')

In [ ]:
# Test synthesis — listen to generated audio
from TTS.api import TTS
from IPython.display import Audio, display
import torch

tts = TTS(
    model_path=best_model,
    config_path=config_file,
    gpu=torch.cuda.is_available(),
)

test_sentences = [
    'नमस्कार, तुम कैसा छा?',
    'गढ़वाल बड़ो सुंदर छ',
    'आज मौसम बड़ो छ',
    'मी घर जान्दू',
    'जय बदरी विशाल',
    'खाणा बड़ो स्वादिष्ट छ',
]

for i, text in enumerate(test_sentences):
    out = f'/content/test_{i+1}.wav'
    tts.tts_to_file(text=text, file_path=out)
    print(f'\n{i+1}. {text}')
    display(Audio(out))

## 7️⃣ Export Model for HuggingFace Deployment

In [ ]:
# Package model files for the HuggingFace Gradio Space
import shutil, json, os

EXPORT_DIR = '/content/garhwali_tts_export'
os.makedirs(EXPORT_DIR, exist_ok=True)

# Copy model + config (names must match tts-training/huggingface/app.py)
shutil.copy(best_model, os.path.join(EXPORT_DIR, 'model.pth'))
shutil.copy(config_file, os.path.join(EXPORT_DIR, 'config.json'))

# Model card
info = {
    'name': 'Garhwali VITS TTS — Ghughuti AI',
    'language': 'Garhwali (गढ़वाली)',
    'script': 'Devanagari',
    'base_model': 'tts_models/en/ljspeech/vits',
    'framework': 'Coqui TTS (VITS)',
    'training_utterances': len(train_lines),
    'eval_utterances': len(eval_lines),
    'total_audio_minutes': round(total_dur / 60, 1),
    'characters': len(chars),
    'epochs': config.epochs,
}
with open(os.path.join(EXPORT_DIR, 'info.json'), 'w') as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

# ZIP for download
shutil.make_archive('/content/garhwali_tts_model', 'zip', EXPORT_DIR)
print('✅ Export ready: /content/garhwali_tts_model.zip')
print(json.dumps(info, indent=2, ensure_ascii=False))

In [ ]:
# Download the ZIP to your machine
from google.colab import files
files.download('/content/garhwali_tts_model.zip')

In [ ]:
# ── OR: Push directly to HuggingFace Space ──
from huggingface_hub import HfApi, login

# Authenticate (paste your HF token when prompted)
login()

api = HfApi()

# Replace with your HF username / space name
HF_REPO = 'your-username/garhwali-tts'

# Upload model files
for fname in ['model.pth', 'config.json', 'info.json']:
    api.upload_file(
        path_or_fileobj=os.path.join(EXPORT_DIR, fname),
        path_in_repo=fname,
        repo_id=HF_REPO,
        repo_type='space',
    )
    print(f'  Uploaded {fname}')

print(f'\n✅ Model uploaded to https://huggingface.co/spaces/{HF_REPO}')
print('   Now upload app.py & requirements.txt from tts-training/huggingface/')

## 📋 Checklist & Troubleshooting

### Before training
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)
- [ ] ≥ 50 recordings downloaded (≥ 30 min audio ideal)
- [ ] metadata.csv has `id|text|text` format
- [ ] All WAVs are 22050 Hz, mono, 16-bit

### Common issues
| Problem | Fix |
|---------|-----|
| **OOM on T4** | Set `batch_size=8` (or 4) in the config cell |
| **Garbled output** | Train longer (increase `epochs`), or record more data |
| **Training loss doesn't drop** | Check audio quality — silence, noise, clipping |
| **Characters missing** | Re-run the charset cell after updating metadata |

### Deploy to HuggingFace
1. Create a new **Gradio** Space at [huggingface.co/new-space](https://huggingface.co/new-space)
2. Upload these files to the Space:
   - `model.pth` & `config.json` (from export)
   - `app.py` & `requirements.txt` (from `tts-training/huggingface/`)
3. The Space auto-deploys and serves the Garhwali TTS API

### Record more data
Visit **pahaditube.in/voice-recording** — the 200+ training sentences are already loaded.
More recordings from more speakers = better model quality.